# Flood Prediction Model Development

This notebook focuses on building a machine learning model to predict flood risks using various environmental and hydrological features.

## Objectives:
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Model Training & Comparison
- Model Evaluation

## 1. Imports

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
import joblib
import warnings
import time
import os

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
print("Libraries loaded successfully!")

Libraries loaded successfully!


## 2. Data Loading

### ✅ STEP 14 — Load Dataset in Notebook

In [25]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/train.csv")

# Show first 5 rows
df.head()

,id,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability
0,0,5,8,5,8,6,4,4,3,3,...,5,3,3,5,4,7,5,7,3,0.445
1,1,6,7,4,4,8,8,3,5,4,...,7,2,0,3,5,3,3,4,3,0.450
2,2,6,5,6,7,3,7,1,5,4,...,7,3,7,5,6,8,2,3,3,0.530
3,3,3,4,6,5,4,8,4,7,6,...,2,4,7,4,4,6,5,7,5,0.535
4,4,5,3,2,6,4,4,3,3,3,...,2,2,6,6,4,1,2,3,5,0.415


## 3. Exploratory Data Analysis (EDA)

### # ============================================
### # COMPLETE DATASET ANALYSIS PIPELINE
### # ============================================

In [26]:
# 1. Show dataset shape
print("===================================")
print("DATASET SHAPE")
print("===================================")
print(df.shape)

# 2. Show column names
print("\n===================================")
print("COLUMN NAMES")
print("===================================")
print(df.columns.tolist())

# 3. Show dataframe info
print("\n===================================")
print("DATAFRAME INFO")
print("===================================")
df.info()

# 4. Show first 5 rows
print("\n===================================")
print("FIRST 5 ROWS")
print("===================================")
display(df.head())

# 5. Statistical summary
print("\n===================================")
print("STATISTICAL SUMMARY")
print("===================================")
display(df.describe())

# 6. Missing values
print("\n===================================")
print("MISSING VALUES")
print("===================================")
missing_values = df.isnull().sum()
print(missing_values)

# 7. Duplicate rows
print("\n===================================")
print("DUPLICATE ROWS")
print("===================================")
print("Duplicates:", df.duplicated().sum())

DATASET SHAPE
(1117957, 22)

COLUMN NAMES
['id', 'MonsoonIntensity', 'TopographyDrainage', 'RiverManagement', 'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality', 'Siltation', 'AgriculturalPractices', 'Encroachments', 'IneffectiveDisasterPreparedness', 'DrainageSystems', 'CoastalVulnerability', 'Landslides', 'Watersheds', 'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss', 'InadequatePlanning', 'PoliticalFactors', 'FloodProbability']

DATAFRAME INFO
<class 'pandas.DataFrame'>
RangeIndex: 1117957 entries, 0 to 1117956
Data columns (total 22 columns):
 #   Column                           Non-Null Count    Dtype  
---  ------                           --------------    -----  
 0   id                               1117957 non-null  int64  
 1   MonsoonIntensity                 1117957 non-null  int64  
 2   TopographyDrainage               1117957 non-null  int64  
 3   RiverManagement                  1117957 non-null  int64  
 4   Deforestation                 

,id,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability
0,0,5,8,5,8,6,4,4,3,3,...,5,3,3,5,4,7,5,7,3,0.445
1,1,6,7,4,4,8,8,3,5,4,...,7,2,0,3,5,3,3,4,3,0.450
2,2,6,5,6,7,3,7,1,5,4,...,7,3,7,5,6,8,2,3,3,0.530
3,3,3,4,6,5,4,8,4,7,6,...,2,4,7,4,4,6,5,7,5,0.535
4,4,5,3,2,6,4,4,3,3,3,...,2,2,6,6,4,1,2,3,5,0.415



STATISTICAL SUMMARY


,id,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability
count,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,...,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06,1.117957e+06
mean,5.589780e+05,4.921450e+00,4.926671e+00,4.955322e+00,4.942240e+00,4.942517e+00,4.934093e+00,4.955878e+00,4.927791e+00,4.942619e+00,...,4.946893e+00,4.953999e+00,4.931376e+00,4.929032e+00,4.925907e+00,4.927520e+00,4.950859e+00,4.940587e+00,4.939004e+00,5.044803e-01
std,3.227265e+05,2.056387e+00,2.093879e+00,2.072186e+00,2.051689e+00,2.083391e+00,2.057742e+00,2.083063e+00,2.065992e+00,2.068545e+00,...,2.072333e+00,2.088899e+00,2.078287e+00,2.082395e+00,2.064813e+00,2.074176e+00,2.068696e+00,2.081123e+00,2.090350e+00,5.102610e-02
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.850000e-01
25%,2.794890e+05,3.000000e+00,3.000000e+00,4.000000e+00,4.000000e+00,3.000000e+00,3.000000e+00,4.000000e+00,3.000000e+00,3.000000e+00,...,4.000000e+00,3.000000e+00,3.000000e+00,3.000000e+00,3.000000e+00,3.000000e+00,4.000000e+00,3.000000e+00,3.000000e+00,4.700000e-01
50%,5.589780e+05,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,...,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.050000e-01
75%,8.384670e+05,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,...,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,5.400000e-01
max,1.117956e+06,1.600000e+01,1.800000e+01,1.600000e+01,1.700000e+01,1.700000e+01,1.700000e+01,1.600000e+01,1.600000e+01,1.600000e+01,...,1.700000e+01,1.700000e+01,1.600000e+01,1.600000e+01,1.700000e+01,1.800000e+01,1.900000e+01,1.600000e+01,1.600000e+01,7.250000e-01



MISSING VALUES
id                                 0
MonsoonIntensity                   0
TopographyDrainage                 0
RiverManagement                    0
Deforestation                      0
Urbanization                       0
ClimateChange                      0
DamsQuality                        0
Siltation                          0
AgriculturalPractices              0
Encroachments                      0
IneffectiveDisasterPreparedness    0
DrainageSystems                    0
CoastalVulnerability               0
Landslides                         0
Watersheds                         0
DeterioratingInfrastructure        0
PopulationScore                    0
WetlandLoss                        0
InadequatePlanning                 0
PoliticalFactors                   0
FloodProbability                   0
dtype: int64

DUPLICATE ROWS
Duplicates: 0


### # ============================================
### # 8. IDENTIFY TARGET + FEATURES
### # ============================================

In [27]:
print("\n===================================")
print("TARGET + FEATURE IDENTIFICATION")
print("===================================")

# Automatically identify target column
possible_targets = [
    col for col in df.columns
    if "flood" in col.lower()
    or "target" in col.lower()
    or "label" in col.lower()
    or "probability" in col.lower()
]

target_column = possible_targets[-1]

feature_columns = [col for col in df.columns if col != target_column]

print("Target Column:")
print(target_column)

print("\nNumber of Features:")
print(len(feature_columns))

print("\nFeature Columns:")
print(feature_columns)


TARGET + FEATURE IDENTIFICATION
Target Column:
FloodProbability

Number of Features:
21

Feature Columns:
['id', 'MonsoonIntensity', 'TopographyDrainage', 'RiverManagement', 'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality', 'Siltation', 'AgriculturalPractices', 'Encroachments', 'IneffectiveDisasterPreparedness', 'DrainageSystems', 'CoastalVulnerability', 'Landslides', 'Watersheds', 'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss', 'InadequatePlanning', 'PoliticalFactors']


### # ============================================
### # 9. DATASET ANALYSIS
### # ============================================

In [28]:
print("\n===================================")
print("ML PROBLEM ANALYSIS")
print("===================================")

# Determine regression/classification
unique_target_values = df[target_column].nunique()

if df[target_column].dtype in ["float64", "float32"] and unique_target_values > 20:
    problem_type = "Regression"
else:
    problem_type = "Classification"

print(f"Problem Type: {problem_type}")

# Check categorical columns
categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("\nCategorical Columns:")
if categorical_columns:
    print(categorical_columns)
else:
    print("No categorical columns found.")

# Missing values
total_missing = missing_values.sum()

print("\nMissing Values:")
print(total_missing)

# Columns that may need scaling
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("\nColumns That May Need Scaling:")
print(numeric_cols)

# Detect suspicious/outlier columns
print("\nPotential Outlier/Suspicious Columns:")

suspicious_cols = []

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    if outliers > 0:
        suspicious_cols.append((col, outliers))

if suspicious_cols:
    for col, count in suspicious_cols:
        print(f"{col}: {count} potential outliers")
else:
    print("No major outliers detected.")


ML PROBLEM ANALYSIS
Problem Type: Regression

Categorical Columns:
No categorical columns found.

Missing Values:
0

Columns That May Need Scaling:
['id', 'MonsoonIntensity', 'TopographyDrainage', 'RiverManagement', 'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality', 'Siltation', 'AgriculturalPractices', 'Encroachments', 'IneffectiveDisasterPreparedness', 'DrainageSystems', 'CoastalVulnerability', 'Landslides', 'Watersheds', 'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss', 'InadequatePlanning', 'PoliticalFactors', 'FloodProbability']

Potential Outlier/Suspicious Columns:
MonsoonIntensity: 9244 potential outliers
TopographyDrainage: 9575 potential outliers
RiverManagement: 29617 potential outliers
Deforestation: 28235 potential outliers
Urbanization: 9184 potential outliers
ClimateChange: 8702 potential outliers
DamsQuality: 31097 potential outliers
Siltation: 9079 potential outliers
AgriculturalPractices: 9006 potential outliers
Encroachments: 31141 poten

### # ============================================
### # 10. PROFESSIONAL SUMMARY
### # ============================================

In [29]:
print("\n===================================")
print("PROFESSIONAL DATASET SUMMARY")
print("===================================")

print(f"""
Dataset Readiness Summary
-------------------------
- Dataset Shape: {df.shape}
- Problem Type: {problem_type}
- Target Column: {target_column}
- Total Features: {len(feature_columns)}
- Missing Values: {total_missing}
- Duplicate Rows: {df.duplicated().sum()}
- Categorical Columns: {len(categorical_columns)}

Assessment:
The dataset appears clean and highly suitable for Machine Learning training.
It is structured, numerical, and optimized for predictive modeling.

Recommended Initial Models:
- Random Forest Regressor
- XGBoost Regressor
- LightGBM Regressor

Next Recommended Steps:
1. Feature-target split
2. Train-test split
3. Feature scaling (if needed)
4. Baseline model training
5. Model evaluation
6. Feature importance analysis
""")


PROFESSIONAL DATASET SUMMARY

Dataset Readiness Summary
-------------------------
- Dataset Shape: (1117957, 22)
- Problem Type: Regression
- Target Column: FloodProbability
- Total Features: 21
- Missing Values: 0
- Duplicate Rows: 0
- Categorical Columns: 0

Assessment:
The dataset appears clean and highly suitable for Machine Learning training.
It is structured, numerical, and optimized for predictive modeling.

Recommended Initial Models:
- Random Forest Regressor
- XGBoost Regressor
- LightGBM Regressor

Next Recommended Steps:
1. Feature-target split
2. Train-test split
3. Feature scaling (if needed)
4. Baseline model training
5. Model evaluation
6. Feature importance analysis



## 4. Feature Engineering

### # ============================================
### # FEATURE-TARGET SEPARATION
### # ============================================

In [30]:
# 1. Automatically identify the target column
target_col = [col for col in df.columns if "probability" in col.lower()][-1]

# 2. Create X (Features) and y (Target)
# We also drop 'id' as it's a unique identifier with no predictive value
X = df.drop(columns=[target_col, 'id'])
y = df[target_col]

# 3. Print shapes and heads
print("X Shape (Features):", X.shape)
print("y Shape (Target):  ", y.shape)

print("\nFirst 5 rows of X:")
display(X.head())

print("\nFirst 5 rows of y:")
display(y.head())

# 4. Verification
if target_col not in X.columns:
    print(f"\n✅ Verification Successful: '{target_col}' has been removed from X.")
else:
    print(f"\n❌ Verification Failed: '{target_col}' is still in X.")

if 'id' not in X.columns:
    print("✅ Verification Successful: 'id' column has been removed from X.")

X Shape (Features): (1117957, 20)
y Shape (Target):   (1117957,)

First 5 rows of X:


,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,Encroachments,IneffectiveDisasterPreparedness,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors
0,5,8,5,8,6,4,4,3,3,4,2,5,3,3,5,4,7,5,7,3
1,6,7,4,4,8,8,3,5,4,6,9,7,2,0,3,5,3,3,4,3
2,6,5,6,7,3,7,1,5,4,5,6,7,3,7,5,6,8,2,3,3
3,3,4,6,5,4,8,4,7,6,8,5,2,4,7,4,4,6,5,7,5
4,5,3,2,6,4,4,3,3,3,3,5,2,2,6,6,4,1,2,3,5



First 5 rows of y:


0    0.445
1    0.450
2    0.530
3    0.535
4    0.415
Name: FloodProbability, dtype: float64


✅ Verification Successful: 'FloodProbability' has been removed from X.
✅ Verification Successful: 'id' column has been removed from X.


### # ============================================
### # TRAIN-TEST SPLIT
### # ============================================

In [31]:
from sklearn.model_selection import train_test_split

# 1. Perform Train-Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 2. Dataset Shape Verification
print("===================================")
print("TRAIN-TEST SPLIT SHAPES")
print("===================================")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

# 3. Verification: training + testing = original size
total_rows = X_train.shape[0] + X_test.shape[0]
print(f"\nTotal rows in split: {total_rows} (Original: {X.shape[0]})")
if total_rows == X.shape[0]:
    print("✅ Verification Successful: Split row count matches original size.")

# 4. Percentage split verification
print(f"\nTraining Split: {len(X_train)/len(X)*100:.1f}%")
print(f"Testing Split:  {len(X_test)/len(X)*100:.1f}%")

TRAIN-TEST SPLIT SHAPES
X_train shape: (894365, 20)
X_test shape:  (223592, 20)
y_train shape: (894365,)
y_test shape:  (223592,)

Total rows in split: 1117957 (Original: 1117957)
✅ Verification Successful: Split row count matches original size.

Training Split: 80.0%
Testing Split:  20.0%


## 5. Model Training & Comparison

### # ============================================
### # BASELINE MODEL: RANDOM FOREST REGRESSOR
### # ============================================

In [32]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

# 1. Initialize the model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Measure Training Time
print("Training the Random Forest Baseline model...")
start_time = time.time()

# 3. Fit the model
rf_model.fit(X_train, y_train)

end_time = time.time()
rf_training_duration = end_time - start_time
print(f"Training Completed in {rf_training_duration:.2f} seconds.")

# 4. Predict on Test Set
y_pred_rf = rf_model.predict(X_test)

# 5. Evaluation Metrics
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_mse = mean_squared_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, y_pred_rf)

print("\n===================================")
print("RANDOM FOREST METRICS")
print("===================================")
print(f"MAE:  {rf_mae:.5f}")
print(f"RMSE: {rf_rmse:.5f}")
print(f"R²:   {rf_r2:.5f}")

Training the Random Forest Baseline model...
Training Completed in 209.72 seconds.

RANDOM FOREST METRICS
MAE:  0.02447
RMSE: 0.02994
R²:   0.65511


### # ============================================
### # ADVANCED MODEL: XGBOOST REGRESSOR
### # ============================================

In [33]:
from xgboost import XGBRegressor

# 1. Initialize XGBoost Model
xgb_model = XGBRegressor(
    n_estimators=300, 
    learning_rate=0.05, 
    max_depth=8, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    random_state=42, 
    n_jobs=-1
)

# 2. Measure Training Time
print("Training the Advanced XGBoost model...")
start_time = time.time()

# 3. Fit the model
xgb_model.fit(X_train, y_train)

end_time = time.time()
xgb_training_duration = end_time - start_time
print(f"Training Completed in {xgb_training_duration:.2f} seconds.")

# 4. Predict on Test Set
y_pred_xgb = xgb_model.predict(X_test)

# 5. Evaluation Metrics
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)
xgb_mse = mean_squared_error(y_test, y_pred_xgb)
xgb_rmse = np.sqrt(xgb_mse)
xgb_r2 = r2_score(y_test, y_pred_xgb)

print("\n===================================")
print("XGBOOST METRICS")
print("===================================")
print(f"MAE:  {xgb_mae:.5f}")
print(f"RMSE: {xgb_rmse:.5f}")
print(f"R²:   {xgb_r2:.5f}")

Training the Advanced XGBoost model...
Training Completed in 17.06 seconds.

XGBOOST METRICS
MAE:  0.01703
RMSE: 0.02124
R²:   0.82637


### # ============================================
### # MODEL PERSISTENCE & INFERENCE
### # ============================================

In [34]:
import joblib
import os

# 1. Create models directory if it doesn't exist
if not os.path.exists('../models'):
    os.makedirs('../models')

# 2. Save the trained XGBoost model
model_path = '../models/xgboost_flood_model.pkl'
joblib.dump(xgb_model, model_path)
print(f"Model saved successfully to: {model_path}")

# 3. Reload the model
loaded_xgb_model = joblib.load(model_path)
print("Model reloaded successfully.")

# 4. Verification: Loaded model works and predictions match
y_pred_loaded = loaded_xgb_model.predict(X_test)
if np.allclose(y_pred_xgb, y_pred_loaded):
    print("✅ Verification Successful: Loaded model predictions match original model.")

# 5. Sample Prediction System
sample_index = 0
sample_row = X_test.iloc[[sample_index]]
actual_prob = y_test.iloc[sample_index]

predicted_prob = loaded_xgb_model.predict(sample_row)[0]

print("\n===================================")
print("SAMPLE PREDICTION INFERENCE")
print("===================================")
print(f"Actual Flood Probability:    {actual_prob:.4f}")
print(f"Predicted Flood Probability: {predicted_prob:.4f}")

# 6. Simple Risk Interpretation System
def get_risk_level(prob):
    if prob <= 0.3:
        return "🟢 LOW RISK"
    elif prob <= 0.6:
        return "🟡 MEDIUM RISK"
    else:
        return "🔴 HIGH RISK"

print(f"Risk Assessment:            {get_risk_level(predicted_prob)}")

Model saved successfully to: ../models/xgboost_flood_model.pkl
Model reloaded successfully.
✅ Verification Successful: Loaded model predictions match original model.

SAMPLE PREDICTION INFERENCE
Actual Flood Probability:    0.5600
Predicted Flood Probability: 0.5338
Risk Assessment:            🟡 MEDIUM RISK


### Professional Explanations:

**1. Why Model Persistence Matters**:
Training a model on 1.1 million rows takes significant time and computational resources. Model persistence allows us to save the trained "brain" of the AI so we can reuse it instantly without retraining. This is critical for production deployment.

**2. How Real-World AI Systems Load Models**:
In production, models are typically saved as artifacts (like `.pkl`, `.h5`, or `.onnx`). Microservices (often built with FastAPI or Flask) load these models once during startup and keep them in memory to serve lightning-fast predictions to users.

**3. Connection to APIs and Live Alerts**:
When new weather data arrives, an API can pass that data through this reloaded model. If the predicted probability exceeds a certain threshold (e.g., 0.6), the system can automatically trigger SMS or mobile alerts to residents and emergency services.